# 03 · Naive Bayes model building

This notebook compares a bounded set of Naive Bayes text pipelines using the repository’s labeled news data. It selects by **mean 5-fold validation macro F1**; this is the predeclared primary metric. It does not inspect or score the final holdout. Notebook 4 will load the exported pipeline and evaluate that holdout once.

## Decisions agreed before modeling

- **Task/labels:** binary classification; label is assigned from the source file (`Fake.csv` → `fake`, `True.csv` → `real`).
- **Inputs:** title plus body text, joined as one document. Exclude `subject`, `date`, and source filename: Notebook 2 found subject categories perfectly aligned with the labels, and date parsing/coverage is strongly label-dependent.
- **Repeated content:** exact copies by title and body are deduplicated before the split. An audit found 5,793 excess copies and no conflicting labels among exact title/body groups.
- **Split:** stratified 80/20 train/holdout with seed 42. Use only the 80% training portion for a stratified 5-fold CV search; reserve the holdout for Notebook 4.
- **Candidate algorithms:** MultinomialNB and ComplementNB. They are candidates supported by nonnegative word count and TF-IDF features, not assumed winners.
- **Representations:** word-count Bag of Words and TF-IDF. Compare both under the same folds.
- **Search budget:** 48 fixed configurations total (2 NB variants × 2 representations × one n-gram range × 2 `min_df` values × 6 `alpha` values), with 5-fold CV (240 configuration-fold evaluations, including the baseline). Use unigrams only because TF-IDF bigrams with `min_df=2` exceeded the previous 60-minute budget. Reuse fold feature matrices to avoid redundant vectorization. Apply a 60-minute budget to baseline plus CV search; time the final full-training refit separately. Do not expand the grid based on observed validation scores.

### Interpretation limits
The source field is confounded with the label. Excluding explicit metadata reduces an obvious shortcut, but article wording can still carry source or time-period signals. This random holdout estimates performance on deduplicated articles from this dataset’s distribution; it does not demonstrate performance for unseen publishers or future news.

## 1. Reproducibility, environment, and seed

The global seed is **42**. Record versions from the active notebook kernel. During CV, each vectorizer is fitted only on its training fold; one transformed fold is reused across NB/alpha choices with the same feature settings. The exported model is a fitted scikit-learn pipeline.

In [ ]:
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
import importlib.metadata as package_metadata
import os
import platform
import random
import time

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
PALETTE = {'real': '#0072B2', 'fake': '#D55E00'}
sns.set_theme(style='whitegrid', context='notebook', font_scale=1.0)
plt.rcParams.update({'figure.dpi': 120, 'axes.titleweight': 'bold', 'axes.spines.top': False, 'axes.spines.right': False, 'grid.alpha': 0.22})

packages_to_record = ['pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'joblib']
versions = {'python': platform.python_version()}
for package in packages_to_record:
    try:
        versions[package] = package_metadata.version(package)
    except package_metadata.PackageNotFoundError:
        versions[package] = 'not installed'
print('Seed:', GLOBAL_SEED)
print('Environment:', versions)
print('Available logical CPUs:', os.cpu_count())

## 2. Load labels, audit exact content duplicates, and prepare text

The duplicate key uses exact title and body strings, not metadata. Check for conflicting labels before deduplicating; stop if any are found. Keep one copy of each article content so exact copies cannot cross the train/holdout or CV folds. Blank body strings are retained because a title may still contain usable text.

In [ ]:
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'data' / 'raw' / 'News').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate data/raw/News from the notebook working directory.')
raw_dir = PROJECT_ROOT / 'data' / 'raw' / 'News'
frames = []
for label, filename in [('fake', 'Fake.csv'), ('real', 'True.csv')]:
    frame = pd.read_csv(raw_dir / filename)
    frame['label'] = label
    frame['source_file'] = filename
    frames.append(frame)
news = pd.concat(frames, ignore_index=True)
for column in ['title', 'text']:
    news[column] = news[column].fillna('').astype(str)

conflicting_content = news.groupby(['title', 'text'], dropna=False)['label'].nunique().gt(1)
conflicting_groups = int(conflicting_content.sum())
print('Raw rows:', f'{len(news):,}')
print('Exact title/body groups with conflicting labels:', conflicting_groups)
if conflicting_groups:
    raise ValueError('Resolve conflicting labels for identical title/body content before modeling.')

duplicate_mask = news.duplicated(subset=['title', 'text'], keep='first')
duplicate_excess = int(duplicate_mask.sum())
news = news.drop_duplicates(subset=['title', 'text'], keep='first').copy().reset_index(drop=True)
news['article_id'] = [
    sha256((title + '\0' + body).encode('utf-8')).hexdigest()
    for title, body in zip(news['title'], news['text'])
]
if not news['article_id'].is_unique:
    raise ValueError('Article key collision or repeated article remained after deduplication.')
news['model_text'] = news['title'] + '\n\n' + news['text']

print('Exact duplicate rows removed:', f'{duplicate_excess:,}')
print('Unique article rows:', f'{len(news):,}')
print('Post-dedup class counts:')
display(news['label'].value_counts().rename_axis('label').to_frame('rows'))
print('Candidate model inputs: title + body text only; subject/date/source_file excluded.')
print('DataFrame memory (MiB):', round(news.memory_usage(deep=True).sum() / 1024**2, 2))

## 3. Make the untouched holdout and declare cross-validation

The stratified 20% holdout is split off before any vectorizer fitting or model selection. It is not used in the baseline or hyperparameter search. The 5-fold stratified splitter is reused for both comparisons to keep them comparable.

In [ ]:
X_all = news['model_text'].reset_index(drop=True)
y_all = news['label'].reset_index(drop=True)
article_ids = news['article_id'].reset_index(drop=True)
all_positions = np.arange(len(news))
train_positions, holdout_positions = train_test_split(
    all_positions, test_size=0.20, stratify=y_all, random_state=GLOBAL_SEED
)
X_train = X_all.iloc[train_positions].reset_index(drop=True)
y_train = y_all.iloc[train_positions].reset_index(drop=True)
holdout_ids = article_ids.iloc[holdout_positions].reset_index(drop=True)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=GLOBAL_SEED)
print(f'Training articles: {len(X_train):,}')
print(f'Reserved holdout articles: {len(holdout_ids):,}')
print('Training class proportions:')
display(y_train.value_counts(normalize=True).rename('proportion').to_frame())
print('Holdout class proportions (counts only; no model evaluation):')
display(y_all.iloc[holdout_positions].value_counts(normalize=True).rename('proportion').to_frame())

## 4. Establish a Naive Bayes baseline

The baseline is deliberately simple and remains within the requested model family: unigram count features with MultinomialNB (`alpha=1`). It is scored on the training portion’s 5 folds. Macro F1 is primary; accuracy, macro precision, and macro recall provide context. The holdout remains untouched.

In [ ]:
baseline_pipeline = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1, 1), min_df=2)),
    ('nb', MultinomialNB(alpha=1.0)),
])
scoring = {
    'macro_f1': 'f1_macro',
    'accuracy': 'accuracy',
    'macro_precision': 'precision_macro',
    'macro_recall': 'recall_macro',
}
baseline_start = time.perf_counter()
baseline_scores = cross_validate(
    baseline_pipeline, X_train, y_train, cv=cv, scoring=scoring,
    return_train_score=True, n_jobs=1, error_score='raise'
)
baseline_seconds = time.perf_counter() - baseline_start
baseline_summary = pd.DataFrame({
    'metric': ['macro_f1', 'accuracy', 'macro_precision', 'macro_recall'],
    'validation_mean': [baseline_scores[f'test_{m}'].mean() for m in scoring],
    'validation_std': [baseline_scores[f'test_{m}'].std(ddof=1) for m in scoring],
    'training_macro_f1_mean': baseline_scores['train_macro_f1'].mean(),
})
display(baseline_summary)
print(f'Baseline 5-fold run time: {baseline_seconds:.1f} seconds')
baseline_config = {
    'model_name': 'MultinomialNB', 'representation': 'CountVectorizer (BoW)',
    'ngram_range': (1, 1), 'min_df': 2, 'alpha': 1.0,
    'validation_macro_f1_mean': float(baseline_scores['test_macro_f1'].mean()),
    'validation_macro_f1_std': float(baseline_scores['test_macro_f1'].std(ddof=1)),
    'validation_accuracy_mean': float(baseline_scores['test_accuracy'].mean()),
    'validation_macro_precision_mean': float(baseline_scores['test_macro_precision'].mean()),
    'validation_macro_recall_mean': float(baseline_scores['test_macro_recall'].mean()),
    'training_macro_f1_mean': float(baseline_scores['train_macro_f1'].mean()),
    'classifier_fit_seconds_mean': float(np.mean(baseline_scores['fit_time'])),
    'is_baseline': True,
}

## 5. Controlled hyperparameter search

Search 48 fixed unigram configurations and reuse each fold’s transformed BoW/TF-IDF matrix across NB variants and `alpha` values. This cell vectorizes each of the 4 feature settings once per fold (20 fold vectorizations), then fits the 12 applicable NB/alpha combinations on that sparse matrix. The baseline configuration is already scored in the preceding cell.

The baseline plus full candidate search has a **60-minute budget**. Progress prints after each feature-setting/fold. If the budget is exceeded, the cell stops without choosing a partial or incomparable winner. The final refit is one additional fit on the training portion and is timed separately.

In [ ]:
from itertools import product

MAX_TRAINING_MINUTES = 60
feature_settings = [
    {'representation': representation, 'ngram_range': ngram_range, 'min_df': min_df}
    for representation, ngram_range, min_df in product(
        ['CountVectorizer (BoW)', 'TF-IDF'], [(1, 1)], [2, 5]
    )
]
candidate_configs = [
    {**feature, 'model_name': model_name, 'alpha': alpha}
    for feature in feature_settings
    for model_name in ['MultinomialNB', 'ComplementNB']
    for alpha in [0.01, 0.1, 0.5, 1.0, 2.0, 10.0]
]
assert len(feature_settings) == 4 and len(candidate_configs) == 48

def config_key(config):
    return (config['representation'], config['ngram_range'], config['min_df'], config['model_name'], config['alpha'])

baseline_key = ('CountVectorizer (BoW)', (1, 1), 2, 'MultinomialNB', 1.0)
if baseline_key not in {config_key(config) for config in candidate_configs}:
    raise AssertionError('The declared baseline is missing from the candidate space.')
candidate_fold_results = {
    config_key(config): {'validation_macro_f1': [], 'validation_accuracy': [],
                        'validation_macro_precision': [], 'validation_macro_recall': [],
                        'training_macro_f1': [], 'classifier_fit_seconds': []}
    for config in candidate_configs if config_key(config) != baseline_key
}
splits = list(cv.split(X_train, y_train))
expected_classifier_fits = len(candidate_configs) * len(splits)
print(f'Fixed budget: {len(candidate_configs)} configurations × {len(splits)} folds = {expected_classifier_fits} classifier evaluations total, including the baseline. Candidate search reuses 20 fold feature matrices; the earlier baseline cell separately vectorizes its 5 folds.')
print('Reuse plan: each transformed training/validation fold is shared across its NB type and alpha values.')

def make_vectorizer(representation, ngram_range, min_df):
    vectorizer_class = CountVectorizer if representation == 'CountVectorizer (BoW)' else TfidfVectorizer
    return vectorizer_class(ngram_range=ngram_range, min_df=min_df)

def make_nb(model_name, alpha):
    model_class = MultinomialNB if model_name == 'MultinomialNB' else ComplementNB
    return model_class(alpha=alpha)

search_start = time.perf_counter()
total_training_seconds = baseline_seconds
completed_classifier_fits = 5  # baseline fits completed in the preceding cell
for setting_index, setting in enumerate(feature_settings, start=1):
    matching_configs = [config for config in candidate_configs
                        if config['representation'] == setting['representation']
                        and config['ngram_range'] == setting['ngram_range']
                        and config['min_df'] == setting['min_df']]
    for fold_index, (train_idx, validation_idx) in enumerate(splits, start=1):
        total_training_seconds = baseline_seconds + (time.perf_counter() - search_start)
        if total_training_seconds >= MAX_TRAINING_MINUTES * 60:
            raise TimeoutError('The 60-minute training budget was reached before all configurations completed; no winner was selected.')
        fold_vectorizer = make_vectorizer(**setting)
        X_fold_train = fold_vectorizer.fit_transform(X_train.iloc[train_idx])
        X_fold_validation = fold_vectorizer.transform(X_train.iloc[validation_idx])
        y_fold_train = y_train.iloc[train_idx]
        y_fold_validation = y_train.iloc[validation_idx]
        for config in matching_configs:
            if config_key(config) == baseline_key:
                continue  # baseline metrics were recorded before the search
            total_training_seconds = baseline_seconds + (time.perf_counter() - search_start)
            if total_training_seconds >= MAX_TRAINING_MINUTES * 60:
                raise TimeoutError('The 60-minute training budget was reached before all configurations completed; no winner was selected.')
            estimator = make_nb(config['model_name'], config['alpha'])
            fit_start = time.perf_counter()
            estimator.fit(X_fold_train, y_fold_train)
            train_prediction = estimator.predict(X_fold_train)
            validation_prediction = estimator.predict(X_fold_validation)
            fit_seconds = time.perf_counter() - fit_start
            result = candidate_fold_results[config_key(config)]
            result['validation_macro_f1'].append(f1_score(y_fold_validation, validation_prediction, average='macro'))
            result['validation_accuracy'].append(accuracy_score(y_fold_validation, validation_prediction))
            result['validation_macro_precision'].append(precision_score(y_fold_validation, validation_prediction, average='macro', zero_division=0))
            result['validation_macro_recall'].append(recall_score(y_fold_validation, validation_prediction, average='macro', zero_division=0))
            result['training_macro_f1'].append(f1_score(y_fold_train, train_prediction, average='macro'))
            result['classifier_fit_seconds'].append(fit_seconds)
            completed_classifier_fits += 1
            total_training_seconds = baseline_seconds + (time.perf_counter() - search_start)
            if total_training_seconds >= MAX_TRAINING_MINUTES * 60:
                raise TimeoutError('The 60-minute training budget was reached before all configurations completed; no winner was selected.')
        total_training_seconds = baseline_seconds + (time.perf_counter() - search_start)
        print(f"Feature setting {setting_index}/{len(feature_settings)} · fold {fold_index}/{len(splits)} complete · {completed_classifier_fits}/{expected_classifier_fits} classifier fits · {total_training_seconds/60:.1f} min total", flush=True)
        del fold_vectorizer, X_fold_train, X_fold_validation, y_fold_train, y_fold_validation

search_seconds = time.perf_counter() - search_start
search_records = []
for config in candidate_configs:
    if config_key(config) == baseline_key:
        continue
    scores = candidate_fold_results[config_key(config)]
    search_records.append({
        **config,
        'ngram_range': config['ngram_range'],
        'validation_macro_f1_mean': float(np.mean(scores['validation_macro_f1'])),
        'validation_macro_f1_std': float(np.std(scores['validation_macro_f1'], ddof=1)),
        'validation_accuracy_mean': float(np.mean(scores['validation_accuracy'])),
        'validation_macro_precision_mean': float(np.mean(scores['validation_macro_precision'])),
        'validation_macro_recall_mean': float(np.mean(scores['validation_macro_recall'])),
        'training_macro_f1_mean': float(np.mean(scores['training_macro_f1'])),
        'classifier_fit_seconds_mean': float(np.mean(scores['classifier_fit_seconds'])),
        'is_baseline': False,
    })
best_config = max([baseline_config, *search_records], key=lambda row: row['validation_macro_f1_mean'])
print(f'All configurations complete. Search elapsed: {search_seconds/60:.1f} min; baseline + search: {(baseline_seconds+search_seconds)/60:.1f} min.')
print('Highest mean 5-fold validation macro F1 configuration:')
display(pd.Series(best_config, name='selected candidate').to_frame())

## 6. Experiment log and stability review

The CSV is the source of truth for all 48 configurations, including the separately run baseline. It includes mean/std validation macro F1, secondary metrics, mean training macro F1, and classifier time. The plot compares training and validation macro F1 for the top 12 candidates; a large gap may indicate overfitting but does not change the primary selection metric.

In [ ]:
timestamp_utc = datetime.now(timezone.utc).isoformat()
records = [{'timestamp_utc': timestamp_utc, **baseline_config}]
for row in search_records:
    record = {'timestamp_utc': timestamp_utc, **row}
    records.append(record)
experiment_log = pd.DataFrame(records).sort_values(
    ['is_baseline', 'validation_macro_f1_mean'], ascending=[False, False]
).reset_index(drop=True)
outputs_dir = PROJECT_ROOT / 'outputs'
outputs_dir.mkdir(parents=True, exist_ok=True)
log_path = outputs_dir / 'experiment_log.csv'
experiment_log.to_csv(log_path, index=False)
print('Saved experiment log:', log_path)
display(experiment_log.head(12))

top = experiment_log.loc[~experiment_log['is_baseline']].head(12).sort_values('validation_macro_f1_mean')
fig, ax = plt.subplots(figsize=(10, 6))
y_positions = np.arange(len(top))
ax.scatter(top['training_macro_f1_mean'], y_positions, color='#999999', label='Training CV mean', s=42, zorder=3)
ax.scatter(top['validation_macro_f1_mean'], y_positions, color='#0072B2', label='Validation CV mean', s=48, zorder=3)
for y, (_, row) in zip(y_positions, top.iterrows()):
    ax.plot([row['validation_macro_f1_mean'], row['training_macro_f1_mean']], [y, y], color='#777777', alpha=.55, linewidth=1)
ax.set_yticks(y_positions, [f"{r.model_name} · {r.representation} · {r.ngram_range} · α={r.alpha:g}" for r in top.itertuples()])
ax.set_xlabel('Macro F1')
ax.set_title('Top validation candidates and their training–validation gap', loc='left', pad=12)
ax.legend(frameon=False, loc='lower right')
sns.despine(ax=ax)
fig.tight_layout()
plt.show()

## 7. Select by the declared metric and export reproducible artifacts

Refit the configuration with the highest mean 5-fold CV macro F1 (including the baseline if it wins) on the full training portion. The untouched holdout is not scored here. Save the complete vectorizer-plus-classifier pipeline and stable IDs for the holdout articles so Notebook 4 can reconstruct the same unseen set. The CSV log records the exact winning row for later justification.

Compressed model artifacts larger than a few MiB are flagged for a commit/LFS/exclude decision; do not silently add a large artifact to Git.

In [ ]:
best_row = experiment_log.loc[experiment_log['validation_macro_f1_mean'].idxmax()]
print('Selected experiment-log row:')
display(best_row.to_frame('value'))
baseline_macro_f1 = float(baseline_scores['test_macro_f1'].mean())
print('Best-minus-baseline validation macro F1:', round(float(best_config['validation_macro_f1_mean'] - baseline_macro_f1), 4))
print('Best candidate fold standard deviation:', round(float(best_config['validation_macro_f1_std']), 4))
print('Best candidate train–validation macro F1 gap:', round(float(best_config['training_macro_f1_mean'] - best_config['validation_macro_f1_mean']), 4))

best_vectorizer = make_vectorizer(best_config['representation'], best_config['ngram_range'], best_config['min_df'])
best_classifier = make_nb(best_config['model_name'], best_config['alpha'])
final_pipeline = Pipeline([('vectorizer', best_vectorizer), ('nb', best_classifier)])
final_fit_start = time.perf_counter()
final_pipeline.fit(X_train, y_train)
final_fit_seconds = time.perf_counter() - final_fit_start
print(f'Final full-training refit time: {final_fit_seconds/60:.2f} minutes')

models_dir = PROJECT_ROOT / 'models'
models_dir.mkdir(parents=True, exist_ok=True)
model_path = models_dir / 'final_naive_bayes_pipeline.joblib'
joblib.dump(final_pipeline, model_path, compress=3)

holdout_path = outputs_dir / 'holdout_article_ids.csv'
pd.DataFrame({'article_id': holdout_ids}).to_csv(holdout_path, index=False)
artifact_mib = model_path.stat().st_size / 1024**2
print('Saved complete pipeline:', model_path)
print(f'Compressed model size: {artifact_mib:.2f} MiB')
print('Saved reserved holdout article IDs:', holdout_path)
if artifact_mib > 5:
    print('REVIEW REQUIRED: model exceeds 5 MiB; decide whether to commit, use Git LFS, or exclude it before staging.')
print('Holdout evaluation: not performed in this notebook.')

## 8. Model-building conclusions and self-critique

After executing the notebook, complete this summary using the experiment log and displayed CV outputs only:

- State the baseline validation macro F1 and its fold standard deviation.
- Name the selected model configuration and cite its row in `experiment_log.csv`.
- Compare selected mean CV macro F1 with baseline; report the selected fold standard deviation and train–validation gap.
- Record runtime, artifact size, and any warnings/errors. Do not claim test performance; Notebook 4 owns the holdout.

### Strongest objection
Even after removing explicit metadata and exact repeated content, a stratified random split can place articles from the same underlying publishers, topics, or periods in both training and validation. The subject/label confounding found in EDA makes source-style shortcuts especially plausible. The fixed holdout and CV can compare NB configurations on this sample, but they cannot prove transfer to unseen publishers or future news. A future dataset with independent source/time labels would be needed for that stronger claim.